In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

perf = spark.table(
    "workspace.default.freddie_perf_raw"
)

survival = spark.table(
    "workspace.default.freddie_survival"
)

cutoff = F.to_date(F.lit("2026-06-01"))

# Parse monthly reporting dates
monthly = perf.select(
    F.col("_c0").alias("loan_id"),
    F.to_date(
        F.concat(F.col("_c1"), F.lit("01")),
        "yyyyMMdd"
    ).alias("report_month"),
    F.col("_c3").alias("delinquency_status")
)

# Check duplicate loan-month keys
duplicates = (
    monthly.groupBy("loan_id", "report_month")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate loan-month groups:", duplicates.count())

# Check invalid dates
print(
    "Invalid reporting dates:",
    monthly.filter(F.col("report_month").isNull()).count()
)

display(duplicates.limit(20))


Duplicate loan-month groups: 0
Invalid reporting dates: 0


loan_id,report_month,count


In [0]:

# Keep reporting months through each loan's exit date
observed = (
    monthly
    .join(
        survival.select("loan_id", "exit_date"),
        "loan_id",
        "inner"
    )
    .filter(
        (F.col("report_month") <= F.col("exit_date")) &
        (F.col("report_month") <= cutoff)
    )
    .select("loan_id", "report_month")
    .distinct()
)

# Order reporting months within each loan
w = Window.partitionBy("loan_id").orderBy("report_month")

continuity = (
    observed
    .withColumn(
        "previous_month",
        F.lag("report_month").over(w)
    )
    .withColumn(
        "month_gap",
        F.months_between(
            "report_month", "previous_month"
        ).cast("int")
    )
    .withColumn(
        "missing_months",
        F.when(
            F.col("month_gap") > 1,
            F.col("month_gap") - 1
        ).otherwise(0)
    )
)

# Summarize at loan level
loan_continuity = (
    continuity.groupBy("loan_id")
    .agg(
        F.sum("missing_months").alias("total_missing_months"),
        F.max("month_gap").alias("max_gap")
    )
)

display(
    loan_continuity.agg(
        F.count("*").alias("total_loans"),
        F.sum(
            (F.col("total_missing_months") > 0).cast("int")
        ).alias("loans_with_gaps"),
        F.sum("total_missing_months").alias("missing_loan_months"),
        F.max("max_gap").alias("maximum_gap_months")
    )
)


total_loans,loans_with_gaps,missing_loan_months,maximum_gap_months
49968,0,0,1


In [0]:

display(
    continuity
    .filter(F.col("month_gap") > 1)
    .select(
        "loan_id",
        "previous_month",
        "report_month",
        "month_gap",
        "missing_months"
    )
    .orderBy(F.desc("month_gap"))
    .limit(30)
)


loan_id,previous_month,report_month,month_gap,missing_months
